# Lab 73: The Cheater Detector

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 8, *The Social Network*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-73.ipynb)

**What you will do:** test whether a logic puzzle about catching cheaters is easier than the same puzzle stated abstractly, compare your results with the book's typical figures, check an AI chatbot for the same split, and see how a fairness-driven responder's earnings compare with a purely rational one in the related Ultimatum Game simulation.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 73 you gave a friend two versions of the same logic puzzle: an abstract version
(letters and numbers, as in Lab 60) and a socially framed version about checking whether
employees who received a bonus worked overtime. Both versions have exactly the same logical
structure, but most people solve the social version far more easily. This gap is the signature
of the **cheater-detection hypothesis** (Cosmides & Tooby, 1992): the brain reasons about rule
violations more effectively when the rule is a social contract than when it is abstract.

## Record your results

Send the abstract version first, then the social version, to a handful of friends (or answer
both yourself). Log whether each person got each version right.

In [ ]:
# example data: replace with yours -- record whether each person answered correctly
wason_df = pd.DataFrame({
    "person": ["You", "Friend 1", "Friend 2", "Friend 3", "Friend 4"],
    "abstract_correct": [False, False, True, False, False],   # letters-and-numbers version
    "social_correct": [True, True, True, False, True],        # employees-and-bonuses version
})
display(wason_df)
pct_abstract = wason_df.abstract_correct.mean() * 100
pct_social = wason_df.social_correct.mean() * 100
pd.Series({"Abstract version": pct_abstract, "Social version": pct_social}).plot.bar(
    color=["grey", "steelblue"], figsize=(5, 3.5))
plt.ylabel("% correct"); plt.ylim(0, 100); plt.xticks(rotation=0); plt.show()
print(f"Abstract version: {pct_abstract:.0f}% correct")
print(f"Social version: {pct_social:.0f}% correct")

## Compare

Lab 60 and Lab 73 report that, across the wider literature, the abstract version is usually
solved correctly by well under a quarter of people (often closer to one in ten), while the
social-contract version is solved correctly by roughly 65-80% (Cosmides, 1989). Check your
own small sample against that pattern -- with only a handful of responses, expect noise.

In [ ]:
print(f"Your group -- abstract version: {pct_abstract:.0f}% correct")
print(f"Your group -- social version: {pct_social:.0f}% correct")
print("Wider literature: often ~10% (well under 25%) correct on the abstract version,")
print("roughly 65-80% on the social version.")
if pct_abstract < 25 and 65 <= pct_social <= 80:
    print("Your group matches the typical pattern closely.")
elif pct_social > pct_abstract:
    print("Your group shows the expected direction (social framing easier), even if the exact")
    print("percentages differ -- a handful of people is a very small, noisy sample.")
else:
    print("Your group does not show the usual social-framing advantage. With only a few")
    print("people this could easily be chance; pool more responses in the class data section below.")

## Simulation

The Wason task above shows that people are highly sensitive to whether someone might be cheating on a stated rule. The Ultimatum Game, introduced by Güth, Schmittberger, and Schwarze (1982), shows a related and more visceral sensitivity to being treated unfairly. One player (the proposer) suggests how to split a sum of money; the other (the responder) can accept the split, or reject it, in which case neither player gets anything. Purely self-interested responders should accept any offer above zero, since something beats nothing—but people routinely reject offers they judge unfair, even though rejecting costs them money. This simulation has two parts: a single-round game comparing a "rational" player (accepts any offer above zero) with a "fairness" player (rejects offers below 30%), and a repeated game in which a proposer adapts its offers to what each type of responder will accept.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n_rounds = 10000

# Part 1: single-round game. Each offer is an independent draw, expressed
# as the percentage of the pot the proposer offers the responder.
offers = np.random.randint(1, 51, n_rounds)

# Rational responder: accepts anything > 0
rational_accept = offers > 0
rational_earnings = np.where(rational_accept, offers, 0)

# Fairness responder: rejects offers below 30%
fairness_threshold = 30
fairness_accept = offers >= fairness_threshold
fairness_earnings = np.where(fairness_accept, offers, 0)

print("Single-round game, offers drawn at random:")
print(f"Rational player: mean earnings = {np.mean(rational_earnings):.1f}, "
      f"accepts {np.mean(rational_accept) * 100:.0f}% of offers")
print(f"Fairness player: mean earnings = {np.mean(fairness_earnings):.1f}, "
      f"accepts {np.mean(fairness_accept) * 100:.0f}% of offers")


# Part 2: repeated game with an adaptive proposer. The proposer starts
# stingy, raises the offer after each rejection, and lowers it slightly
# after each acceptance -- a simple rule for learning what a responder
# will accept.
def adaptive_proposer_game(responder_threshold, n_rounds=2000,
                            start_offer=5, step=1):
    offer = start_offer
    offer_history = np.zeros(n_rounds)
    for i in range(n_rounds):
        offer_history[i] = offer
        accepted = offer >= responder_threshold
        if accepted:
            offer = max(1, offer - step)
        else:
            offer = min(50, offer + step)
    return offer_history


rational_history = adaptive_proposer_game(responder_threshold=1)
fairness_history = adaptive_proposer_game(
    responder_threshold=fairness_threshold)

print("\nRepeated game with an adaptive proposer "
      "(mean offer, last 500 rounds):")
print(f"Facing a rational responder: {rational_history[-500:].mean():.1f}%")
print(f"Facing a fairness responder: {fairness_history[-500:].mean():.1f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(offers, bins=range(0, 52, 2), color="steelblue", alpha=0.7,
    edgecolor="white")
ax1.axvline(fairness_threshold, color="red", linestyle="--",
    linewidth=2, label=f"Fairness threshold ({fairness_threshold}%)")
ax1.set_xlabel("Offer (%)")
ax1.set_ylabel("Frequency")
ax1.set_title("Single round: distribution of offers")
ax1.legend()

ax2.plot(rational_history, color="grey", alpha=0.8,
    label="Facing a rational responder")
ax2.plot(fairness_history, color="steelblue", alpha=0.8,
    label="Facing a fairness responder")
ax2.set_xlabel("Round")
ax2.set_ylabel("Offer (%)")
ax2.set_title("Repeated game: an adaptive proposer learns")
ax2.legend()

plt.tight_layout()
plt.show()

**What to notice:** In the single-round game, the rational player accepts 100% of offers for a mean of about 26 points per round, against the fairness player accepting only around 42% of offers (everything at or above the 30% threshold) for a lower mean of about 17 points per round. In a single round, rejecting unfair offers is a net loss—the fairness player leaves money on the table. The repeated game shows why responders reject anyway: facing a purely rational responder, the adaptive proposer's offers drift down to the minimum (about 1%), because there is never any cost to lowballing; facing a fairness responder, the same proposer's offers climb and settle right around the 30% rejection threshold, because any lower offer keeps getting punished. Costly punishment is individually irrational in any single round, but it is what teaches the other side to stop lowballing—which is also what the wider experimental literature on repeated ultimatum and public-goods games finds: giving people the option to punish free riders, even at a cost to themselves, sustains much higher cooperation than when punishment is unavailable (Fehr & Gächter, 2002).

## Change the parameters

**fairness_threshold** is the minimum percentage of the pot the fairness player will
accept. **max_offer** caps how generous a proposer can ever be. **n_rounds** controls how
many simulated rounds are averaged.
(If the sliders do not appear, run the cell again.)

In [ ]:
from ipywidgets import interact, IntSlider

def explore(fairness_threshold=30, max_offer=50, n_rounds=10000):
    np.random.seed(42)
    offers = np.random.randint(1, max_offer + 1, n_rounds)

    rational_accept = offers > 0
    rational_earnings = np.where(rational_accept, offers, 0)

    fairness_accept = offers >= fairness_threshold
    fairness_earnings = np.where(fairness_accept, offers, 0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.hist(offers, bins=range(0, max_offer + 2, 2), color="steelblue", alpha=0.7, edgecolor="white")
    ax1.axvline(fairness_threshold, color="red", linestyle="--", linewidth=2, label=f"Fairness threshold ({fairness_threshold}%)")
    ax1.set_xlabel("Offer (%)")
    ax1.set_ylabel("Frequency")
    ax1.set_title("Distribution of Offers")
    ax1.legend()

    labels = ["Rational\n(accept all)", "Fairness\n(reject below threshold)"]
    means = [np.mean(rational_earnings), np.mean(fairness_earnings)]
    accept_rates = [np.mean(rational_accept) * 100, np.mean(fairness_accept) * 100]
    bars = ax2.bar(labels, means, color=["grey", "steelblue"], alpha=0.7)
    for bar, rate in zip(bars, accept_rates):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, f"Accept: {rate:.0f}%", ha="center", fontsize=9)
    ax2.set_ylabel("Mean earnings per round")
    ax2.set_title("Earnings: Rationality vs. Fairness")
    plt.tight_layout()
    plt.show()

    print(f"Rational player: mean earnings = {np.mean(rational_earnings):.1f}, accepts {np.mean(rational_accept) * 100:.0f}% of offers")
    print(f"Fairness player: mean earnings = {np.mean(fairness_earnings):.1f}, accepts {np.mean(fairness_accept) * 100:.0f}% of offers")

interact(explore,
         fairness_threshold=IntSlider(value=30, min=5, max=50, step=5),
         max_offer=IntSlider(value=50, min=20, max=50, step=5),
         n_rounds=IntSlider(value=10000, min=1000, max=20000, step=1000));

## The AI side

The book's "AI extension" suggests testing a chatbot on both framings for consistency. Below,
a small open instruction-tuned model answers the abstract and social versions of the same
puzzle, so you can check whether it treats them the same way -- unlike the split seen in
human performance above.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
from transformers import pipeline
solver = pipeline("text2text-generation", model="google/flan-t5-small")

abstract_prompt = (
    "Cards show A, K, 4, 7, one side visible. Rule: if a card has a vowel on one side, then "
    "it has an even number on the other side. Which cards must you turn over to check the "
    "rule? Answer with card labels only."
)
social_prompt = (
    "Company rule: if an employee got a bonus, they must have worked overtime. Employee A "
    "got a bonus. Employee B did not get a bonus. Employee C worked overtime. Employee D did "
    "not work overtime. Which employees' records must you check? Answer with employee "
    "labels only."
)
abstract_answer = solver(abstract_prompt, max_new_tokens=20)[0]["generated_text"]
social_answer = solver(social_prompt, max_new_tokens=20)[0]["generated_text"]
print("Abstract version -- model answered:", abstract_answer, "(correct: A and 7)")
print("Social version   -- model answered:", social_answer, "(correct: A and D)")

abstract_ok = "A" in abstract_answer and "7" in abstract_answer
social_ok = "A" in social_answer and "D" in social_answer
print(f"\nModel correct on abstract version: {abstract_ok}")
print(f"Model correct on social version: {social_ok}")
print("Consistent" if abstract_ok == social_ok else "Split, like the human pattern above")

flan-t5-small is far smaller and weaker than the large chatbots (GPT-4, Claude, and similar)
the book has in mind, so treat its answers as an illustration of the method, not a strong
replication -- a model this small may well fail both prompts, which tells you little either
way. As Lab 60 discusses, many current large chatbots solve the abstract version about as
well as the social version, unlike the sharp human split shown above. That is itself
informative: whatever lets a capable model succeed on the abstract version -- broad exposure
to formal logic during training, or recognising it as a well-known puzzle -- is evidently not
the same evolved shortcut that helps humans with the social version. It remains debated how
much of any model's success here reflects step-by-step reasoning versus recognising a
familiar problem type from training data.

## Pool the class data

Pool percent correct across the class for both versions to see the social-contract advantage
on a larger, less noisy sample.

In [ ]:
import io
csv_text = """id,abstract_correct,social_correct
P01,0,1
P02,0,1
P03,1,1
P04,0,0
P05,0,1
P06,0,1
P07,1,1
P08,0,0
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
diff = class_df.social_correct - class_df.abstract_correct
boot = [diff.sample(len(diff), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class: {class_df.abstract_correct.mean() * 100:.0f}% correct on the abstract version,")
print(f"       {class_df.social_correct.mean() * 100:.0f}% correct on the social version.")
print(f"Mean advantage of the social framing: {diff.mean() * 100:.0f} percentage points",
      f"(95% bootstrap interval {lo * 100:.0f} to {hi * 100:.0f})")
class_df[["abstract_correct", "social_correct"]].mean().mul(100).plot.bar(
    color=["grey", "steelblue"], figsize=(5, 3.5))
plt.ylabel("% correct"); plt.ylim(0, 100); plt.xticks(rotation=0); plt.show()

## Questions to think about

1. Lower fairness_threshold to 5. Does the fairness player's mean earnings now approach the rational player's? Roughly what threshold value makes the two strategies earn about the same in this single-round game?
2. Raise fairness_threshold to 50, so only an exact even split is accepted. What happens to the fairness player's earnings and acceptance rate, and why might being "too" fair start to cost more than it protects?
3. The main simulation's repeated-game panel shows one proposer facing one fixed responder type at a time (all-rational, or all-fairness). In the offer_history plots, roughly how many rounds does it take the proposer to settle near its final offer against each type, and why does it settle near 1% against the rational responder but near 30% (not higher) against the fairness responder?
4. Reduce max_offer to 30 in the `explore` slider version, so proposers never offer more than 30 percent of the pot. Does the cost of being a fairness player change? What does that reveal about how proposer behaviour and responder strategy depend on each other?

## Challenge

The main simulation's adaptive proposer always faces one fixed type of responder for an
entire run: either purely rational, or purely fairness-minded. Real populations are mixed.
Extend `adaptive_proposer_game` so that, each round, the responder is drawn at random from a
population that is a mix of the two types (for example, 50% rational and 50% fairness), and
see where the proposer's offers settle -- above, below, or between the two single-type
equilibria you saw above.

In [ ]:
# Extends adaptive_proposer_game (defined in the main simulation above) to a *mixed*
# population of responders, instead of one fixed type per run.
def adaptive_proposer_mixed(p_fair, fairness_threshold=30, n_rounds=3000,
                             start_offer=5, step=1):
    offer = start_offer
    history = np.zeros(n_rounds)
    for i in range(n_rounds):
        history[i] = offer
        # each round, draw a responder: fairness-minded with probability p_fair,
        # otherwise purely rational
        is_fair_responder = np.random.random() < p_fair
        threshold = fairness_threshold if is_fair_responder else 1
        accepted = offer >= threshold
        if accepted:
            offer = max(1, offer - step)
        else:
            offer = min(50, offer + step)
    return history

np.random.seed(3)
mix_levels = [0.2, 0.5, 0.8]
plt.figure(figsize=(6, 3.5))
for p_fair in mix_levels:
    history = adaptive_proposer_mixed(p_fair=p_fair)
    plt.plot(history, alpha=0.8, label=f"{p_fair:.0%} fairness responders")
    print(f"{p_fair:.0%} fairness responders: mean offer, last 500 rounds = "
          f"{history[-500:].mean():.1f}%")
plt.axhline(30, color="red", linestyle="--", label="Fairness threshold")
plt.xlabel("Round")
plt.ylabel("Offer (%)")
plt.legend()
plt.title("Adaptive proposer facing a mixed population of responders")
plt.show()

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-73.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")